# Notebook 8 — Forecast-Driven Battery Dispatch

## Overview

Notebook 7 measures the historical upper bound on energy arbitrage. This notebook asks how much of that value can be captured when dispatch decisions are based only on information available at a defined forecast origin.

The workflow is:

$$
\text{forecast-origin data}
\rightarrow
\text{multi-horizon price forecasts}
\rightarrow
\text{receding-horizon optimization}
\rightarrow
\text{realized profit}
$$

The primary outcome is decision value rather than price accuracy alone. Forecasts are evaluated using both statistical errors and the degradation-adjusted battery profit they produce.

In [1]:
# ---------------------------------------------------------------------
# Imports and execution controls
# ---------------------------------------------------------------------

from dataclasses import replace
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from config import MASTER_PARQUET, RANDOM_SEED
from optimization.battery_dispatch import (
    BatteryConfig,
    REFERENCE_BATTERY,
    optimize_battery_dispatch,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

FORECAST_HORIZON_HOURS = 24
EVALUATION_YEARS = list(range(2020, 2026))

# The full walk-forward study fits one model for every year-lead combination
# and then solves an hourly receding-horizon dispatch problem. Keep these
# disabled while editing or smoke-testing individual functions.
RUN_WALK_FORWARD = True
RUN_ROBUSTNESS = False


## 1. Forecast-origin contract

For a target hour $t$ and forecast lead $h$, the model may use:

- calendar information known for the target hour;
- observations available no later than the forecast origin $t-h$;
- forecasts whose historical issue and revision times have been verified.

The model may not use:

- realized target-hour load or net load;
- realized target-hour generation;
- realized outages or imports;
- target-hour ERA5 reanalysis;
- load forecast error;
- pool-price forecast error;
- any feature constructed using the realized target price.

ERA5 remains valid for retrospective mechanism analysis but is not a historical weather forecast.

In [2]:
# ---------------------------------------------------------------------
# Feature-timing register
# ---------------------------------------------------------------------

TARGET_CALENDAR_FEATURES = [
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "is_weekend",
    "is_holiday",
]

ORIGIN_STATE_FEATURES = [
    "pool_price_lag_1h",
    "pool_price_lag_24h",
    "pool_price_lag_168h",
    "pool_price_mean_prior_24h",
    "pool_price_std_prior_24h",
    "net_load_mw_lag_1h",
    "total_outage_mw_lag_1h",
    "net_imports_mw_lag_1h",
    "gas_price_lag_1h",
]

PROHIBITED_FEATURES = [
    "pool_price",
    "ail_mw",
    "net_load_mw",
    "total_outage_mw",
    "net_imports_mw",
    "wind_system_generation",
    "solar_system_generation",
    "aeso_load_forecast_error_mw",
    "aeso_pool_price_forecast_error_cad_mwh",
]

feature_timing_register = pd.DataFrame([
    {
        "family": "Target calendar",
        "status": "Forecast safe",
        "use": "Core model",
    },
    {
        "family": "Lagged market state",
        "status": "Requires publication-lag audit",
        "use": "Core model",
    },
    {
        "family": "AESO load forecast",
        "status": "Issue/revision timing unverified",
        "use": "Hold out initially",
    },
    {
        "family": "AESO pool-price forecast",
        "status": "Issue/revision timing unverified",
        "use": "External benchmark only",
    },
    {
        "family": "ERA5 weather",
        "status": "Reanalysis",
        "use": "Not forecast safe",
    },
    {
        "family": "Same-hour system state",
        "status": "Realized",
        "use": "Prohibited",
    },
])

display(feature_timing_register)

,family,status,use
0,Target calendar,Forecast safe,Core model
1,Lagged market state,Requires publication-lag audit,Core model
2,AESO load forecast,Issue/revision timing unverified,Hold out initially
3,AESO pool-price forecast,Issue/revision timing unverified,External benchmark only
4,ERA5 weather,Reanalysis,Not forecast safe
5,Same-hour system state,Realized,Prohibited


## 2. Forecasting dataset

The first model uses a conservative information set:

- target-hour calendar fields;
- lagged prices and volatility;
- lagged net load;
- lagged outages;
- lagged imports;
- lagged gas prices.

AESO forecasts are excluded from the core model until their historical publication timing is verified.

In [3]:
# ---------------------------------------------------------------------
# Load and validate forecasting inputs
# ---------------------------------------------------------------------

required_columns = [
    "timestamp_utc",
    "timestamp_alberta",
    "year_alberta",
    "pool_price",
    *TARGET_CALENDAR_FEATURES,
    *ORIGIN_STATE_FEATURES,
    "aeso_forecast_pool_price_cad_mwh",
]

master = pd.read_parquet(MASTER_PARQUET, columns=required_columns)
master = (master.loc[master["year_alberta"].between(2015, 2025)].sort_values("timestamp_utc").reset_index(drop=True))

assert not master["timestamp_utc"].duplicated().any()
assert master["timestamp_utc"].is_monotonic_increasing

model_features = (TARGET_CALENDAR_FEATURES + ORIGIN_STATE_FEATURES)

assert not set(
    PROHIBITED_FEATURES
).intersection(model_features)

## 3. Direct multi-horizon targets

Train a separate direct model for each lead from 1 through 24 hours.

For a target hour $t$:

- target-calendar fields remain aligned with $t$;
- market-state fields are aligned with the applicable forecast origin;
- the target is the realized pool price at $t$.

Direct models avoid recursively feeding previous predictions into later forecast leads.

In [4]:
# ---------------------------------------------------------------------
# Construct a leakage-controlled frame for one forecast lead
# ---------------------------------------------------------------------

def build_direct_horizon_frame(
    source,
    lead_hours,
    publication_delay_hours=0,
):
    if lead_hours < 1:
        raise ValueError("lead_hours must be at least one.")
    if publication_delay_hours < 0:
        raise ValueError("publication_delay_hours cannot be negative.")

    frame = source[
        [
            "timestamp_utc",
            "timestamp_alberta",
            "year_alberta",
            "pool_price",
            *TARGET_CALENDAR_FEATURES,
        ]
    ].copy()

    frame = frame.rename(columns={
        "timestamp_utc": "target_timestamp_utc",
        "timestamp_alberta": "target_timestamp_alberta",
        "pool_price": "target_pool_price",
    })

    frame["forecast_origin_utc"] = (
        frame["target_timestamp_utc"]
        - pd.Timedelta(hours=lead_hours)
    )
    frame["forecast_origin_year_alberta"] = (
        frame["forecast_origin_utc"]
        .dt.tz_convert("America/Edmonton")
        .dt.year
    )
    frame["lead_hours"] = lead_hours

    # Every origin-state field already excludes the target hour by one hour.
    # Shift by the remaining lead and any explicit publication delay so each
    # value is no newer than the information cutoff at the forecast origin.
    additional_shift = lead_hours - 1 + publication_delay_hours
    for feature in ORIGIN_STATE_FEATURES:
        frame[feature] = source[feature].shift(additional_shift)

    required = [
        "target_pool_price",
        *TARGET_CALENDAR_FEATURES,
        *ORIGIN_STATE_FEATURES,
    ]
    return (
        frame
        .dropna(subset=required)
        .reset_index(drop=True)
    )


lead_one_frame = build_direct_horizon_frame(master, lead_hours=1)

timing_audit = pd.Series({
    "minimum_lead_hours": (
        (
            lead_one_frame["target_timestamp_utc"]
            - lead_one_frame["forecast_origin_utc"]
        )
        .dt.total_seconds()
        .div(3600)
        .min()
    ),
    "target_after_origin": (
        lead_one_frame["target_timestamp_utc"]
        .gt(lead_one_frame["forecast_origin_utc"])
        .all()
    ),
    "duplicate_target_origin_pairs": (
        lead_one_frame[
            ["target_timestamp_utc", "forecast_origin_utc"]
        ]
        .duplicated()
        .sum()
    ),
})

display(timing_audit.to_frame("value"))
display(lead_one_frame.head())


,value
minimum_lead_hours,1.0
target_after_origin,True
duplicate_target_origin_pairs,0


,target_timestamp_utc,target_timestamp_alberta,year_alberta,target_pool_price,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,is_weekend,is_holiday,forecast_origin_utc,forecast_origin_year_alberta,lead_hours,pool_price_lag_1h,pool_price_lag_24h,pool_price_lag_168h,pool_price_mean_prior_24h,pool_price_std_prior_24h,net_load_mw_lag_1h,total_outage_mw_lag_1h,net_imports_mw_lag_1h,gas_price_lag_1h
0,2017-12-05 08:00:00+00:00,2017-12-05 01:00:00-07:00,2017,15.41,0.258819,0.965926,0.781831,0.62349,-0.5,0.866025,0,0,2017-12-05 07:00:00+00:00,2017,1,15.41,19.17,15.78,19.513333,2.746141,8293.717066,2403.3,-100.0,1.865
1,2017-12-05 09:00:00+00:00,2017-12-05 02:00:00-07:00,2017,15.40,0.500000,0.866025,0.781831,0.62349,-0.5,0.866025,0,0,2017-12-05 08:00:00+00:00,2017,1,15.41,18.50,15.41,19.356667,2.870995,8007.976079,2424.3,-300.0,1.865
2,2017-12-05 10:00:00+00:00,2017-12-05 03:00:00-07:00,2017,15.40,0.707107,0.707107,0.781831,0.62349,-0.5,0.866025,0,0,2017-12-05 09:00:00+00:00,2017,1,15.40,18.50,15.41,19.227500,2.978919,7968.410545,2432.3,-300.0,1.865
3,2017-12-05 11:00:00+00:00,2017-12-05 04:00:00-07:00,2017,15.41,0.866025,0.500000,0.781831,0.62349,-0.5,0.866025,0,0,2017-12-05 10:00:00+00:00,2017,1,15.40,19.95,15.41,19.098333,3.077415,7990.596061,2411.3,-300.0,1.865
4,2017-12-05 12:00:00+00:00,2017-12-05 05:00:00-07:00,2017,17.22,0.965926,0.258819,0.781831,0.62349,-0.5,0.866025,0,0,2017-12-05 11:00:00+00:00,2017,1,15.41,19.50,15.41,18.909167,3.161183,8127.379338,2435.3,-300.0,1.865


## 4. Forecast benchmarks

Begin with only two learned models:

1. regularized linear regression;
2. compact histogram gradient boosting.

The linear model establishes whether the forecast requires meaningful nonlinear structure. Avoid broad model searches until the timing contract and dispatch backtest are validated.



In [5]:
# ---------------------------------------------------------------------
# Define forecast models
# ---------------------------------------------------------------------

MODEL_FEATURES = TARGET_CALENDAR_FEATURES + ORIGIN_STATE_FEATURES


def make_linear_model():
    return Pipeline([
        (
            "impute",
            SimpleImputer(strategy="median"),
        ),
        (
            "scale",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(alpha=10.0),
        ),
    ])


def make_boosted_model():
    return Pipeline([
        (
            "impute",
            SimpleImputer(strategy="median"),
        ),
        (
            "model",
            HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=0.05,
                max_iter=250,
                max_leaf_nodes=15,
                min_samples_leaf=100,
                l2_regularization=1.0,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


# Smoke-test both factories immediately so API or spelling errors fail before
# the expensive walk-forward loop begins.
model_factory_audit = pd.Series({
    "linear_pipeline_created": isinstance(make_linear_model(), Pipeline),
    "boosted_pipeline_created": isinstance(make_boosted_model(), Pipeline),
})
display(model_factory_audit.to_frame("passed"))


,passed
linear_pipeline_created,True
boosted_pipeline_created,True


## 5. Expanding-window evaluation

For each evaluation year:

1. train using all prior years;
2. select model settings using training years only;
3. generate forecasts for the evaluation year;
4. repeat separately for each forecast lead.

The 2020–2025 folds are chronological pseudo-out-of-period evaluations because those years have already been examined elsewhere in the project. They are not untouched operational holdouts.

In [6]:
# ---------------------------------------------------------------------
# Generate direct forecasts by lead and evaluation year
# ---------------------------------------------------------------------

def run_direct_forecast_evaluation(
    source,
    model_factory,
    model_name,
    max_lead_hours=FORECAST_HORIZON_HOURS,
    training_window_years=None,
    publication_delay_hours=0,
):
    prediction_frames = []
    metric_rows = []

    for lead in range(1, max_lead_hours + 1):
        frame = build_direct_horizon_frame(
            source,
            lead,
            publication_delay_hours=publication_delay_hours,
        )

        for evaluation_year in EVALUATION_YEARS:
            train_mask = frame["year_alberta"].lt(evaluation_year)
            if training_window_years is not None:
                train_mask &= frame["year_alberta"].ge(
                    evaluation_year - training_window_years
                )

            # Evaluation years are assigned by forecast origin, not target.
            # This prevents year-boundary forecasts from using observations
            # that occur after their origin.
            test_mask = frame[
                "forecast_origin_year_alberta"
            ].eq(evaluation_year)

            train = frame.loc[train_mask]
            test = frame.loc[test_mask]
            if train.empty or test.empty:
                continue

            model = model_factory()
            model.fit(
                train[MODEL_FEATURES],
                train["target_pool_price"],
            )
            prediction = model.predict(test[MODEL_FEATURES])

            prediction_frames.append(
                pd.DataFrame({
                    "model": model_name,
                    "forecast_origin_utc": test[
                        "forecast_origin_utc"
                    ].to_numpy(),
                    "target_timestamp_utc": test[
                        "target_timestamp_utc"
                    ].to_numpy(),
                    "evaluation_year": evaluation_year,
                    "lead_hours": lead,
                    "actual_price": test[
                        "target_pool_price"
                    ].to_numpy(),
                    "forecast_price": prediction,
                })
            )

            metric_rows.append({
                "model": model_name,
                "evaluation_year": evaluation_year,
                "lead_hours": lead,
                "hours": len(test),
                "mae": mean_absolute_error(
                    test["target_pool_price"],
                    prediction,
                ),
                "rmse": mean_squared_error(
                    test["target_pool_price"],
                    prediction,
                ) ** 0.5,
            })

    if not prediction_frames:
        raise RuntimeError("No forecast predictions were generated.")

    return (
        pd.concat(prediction_frames, ignore_index=True),
        pd.DataFrame(metric_rows),
    )


if RUN_WALK_FORWARD:
    linear_predictions, linear_metrics = (
        run_direct_forecast_evaluation(
            master,
            make_linear_model,
            model_name="Linear",
        )
    )

    boosted_predictions, boosted_metrics = (
        run_direct_forecast_evaluation(
            master,
            make_boosted_model,
            model_name="Boosted",
        )
    )

    forecast_metrics = pd.concat(
        [linear_metrics, boosted_metrics],
        ignore_index=True,
    )

    display(
        forecast_metrics
        .groupby(["model", "evaluation_year"])[["mae", "rmse"]]
        .mean()
        .round(2)
    )
else:
    print(
        "Walk-forward models are configured but not run. "
        "Set RUN_WALK_FORWARD = True to fit all year-lead models."
    )


mae    rmse
model   evaluation_year               
Boosted 2020             17.38   85.16
        2021             52.68  132.31
        2022             88.31  184.39
        2023             86.66  177.19
        2024             40.76  108.85
        2025             25.82   81.69
Linear  2020             26.02   73.29
        2021             60.10  124.47
        2022             95.38  166.06
        2023             99.78  168.02
        2024             56.93  107.41
        2025             45.19   87.50

## 6. Forecast diagnostics

Average error is not sufficient for storage dispatch. Evaluate forecast performance by:

- lead time;
- evaluation year;
- ordinary versus extreme prices;
- scarcity onset versus continuation;
- direction of price change;
- highest-value charging and discharging opportunities.

A model can have lower MAE while producing less arbitrage value.

In [7]:
# ---------------------------------------------------------------------
# Error by lead, model, and realized price regime
# ---------------------------------------------------------------------

if RUN_WALK_FORWARD:
    diagnostics = pd.concat(
        [linear_predictions, boosted_predictions],
        ignore_index=True,
    )

    diagnostics["forecast_error"] = (
        diagnostics["actual_price"]
        - diagnostics["forecast_price"]
    )
    diagnostics["absolute_error"] = diagnostics[
        "forecast_error"
    ].abs()

    diagnostics["price_regime"] = pd.cut(
        diagnostics["actual_price"],
        bins=[-np.inf, 100, 500, 900, np.inf],
        labels=["<100", "100–500", "500–900", "900+"],
    )

    error_by_regime = (
        diagnostics
        .groupby(
            ["model", "lead_hours", "price_regime"],
            observed=True,
        )["absolute_error"]
        .mean()
        .unstack()
    )

    display(error_by_regime.round(2))


price_regime         <100  100–500  500–900    900+
model   lead_hours                                 
Boosted 1            8.26   100.50   227.83  168.61
        2           12.38   118.11   354.39  359.43
        3           14.02   123.15   434.27  505.95
        4           14.21   127.49   481.07  619.16
        5           14.05   131.39   513.06  682.54
        6           14.13   134.35   535.24  726.81
        7           14.35   137.29   544.07  745.97
        8           15.01   137.26   549.97  758.24
        9           15.26   138.50   555.28  775.67
        10          15.28   139.78   560.10  788.00
        11          15.56   139.38   555.86  783.47
        12          16.35   137.65   567.15  802.23
        13          15.73   139.18   561.56  796.12
        14          15.41   139.09   567.46  803.81
        15          15.52   140.36   562.73  799.26
        16          14.96   141.20   572.88  808.34
        17          15.20   141.69   576.19  815.33
        18          15.10   141.84   578.52  817.86
        19          14.79   141.06   573.79  812.79
        20          15.01   141.45   576.99  818.13
        21          15.29   140.87   572.13  810.11
        22          15.18   142.15   571.94  795.42
        23          14.57   144.11   573.32  790.13
        24          14.32   143.77   576.01  803.84
Linear  1           15.17    93.47   196.56  187.41
        2           24.74   107.88   312.53  362.11
        3           30.94   110.47   384.58  484.52
        4           34.58   111.23   424.38  567.45
        5           36.72   110.81   447.92  618.44
        6           37.83   110.84   459.77  649.42
        7           38.33   111.06   465.95  664.50
        8           38.35   111.59   469.08  673.63
        9           38.45   112.08   470.34  677.64
        10          38.58   112.63   472.41  681.03
        11          38.81   112.82   475.02  686.20
        12          39.09   112.63   478.16  692.33
        13          39.33   112.60   481.60  697.11
        14          39.73   112.30   484.60  702.14
        15          40.21   111.77   487.40  707.01
        16          40.65   111.21   489.78  712.57
        17          40.91   111.09   491.92  718.43
        18          40.89   111.41   493.62  723.93
        19          40.39   112.06   494.25  727.59
        20          39.40   112.79   492.83  723.03
        21          38.10   113.50   487.98  713.77
        22          36.58   113.94   479.19  698.05
        23          35.19   114.56   470.30  675.87
        24          34.63   114.51   467.01  667.10

## 7. Receding-horizon dispatch

At each forecast origin:

1. forecast the next 24 hourly prices;
2. optimize dispatch across that horizon;
3. implement only the first decision;
4. settle that decision against the realized pool price;
5. update SOC;
6. advance by one hour and repeat.

Forecast prices determine the decision. Realized prices determine the resulting profit.

In [8]:
# ---------------------------------------------------------------------
# Receding-horizon dispatch
# ---------------------------------------------------------------------

def run_receding_horizon(
    predictions,
    actual_prices,
    base_config,
    horizon_hours=FORECAST_HORIZON_HOURS,
    terminal_soc_policy="hold_current",
    annual_cycle_limit=None,
):
    if terminal_soc_policy not in {"hold_current", "half_full"}:
        raise ValueError(
            "terminal_soc_policy must be 'hold_current' or 'half_full'."
        )

    forecast_panel = (
        predictions.loc[
            predictions["lead_hours"].between(1, horizon_hours)
        ]
        .pivot(
            index="forecast_origin_utc",
            columns="lead_hours",
            values="forecast_price",
        )
        .reindex(columns=range(1, horizon_hours + 1))
        .sort_index()
        .dropna()
    )

    soc = base_config.resolved_initial_soc
    cycle_year = None
    discharged_mwh_ytd = 0.0
    records = []

    for origin, forecast_row in forecast_panel.iterrows():
        target = origin + pd.Timedelta(hours=1)
        if target not in actual_prices.index:
            continue

        target_year = target.tz_convert("America/Edmonton").year
        if target_year != cycle_year:
            cycle_year = target_year
            discharged_mwh_ytd = 0.0

        terminal_soc = (
            soc
            if terminal_soc_policy == "hold_current"
            else 0.5 * base_config.energy_mwh
        )
        horizon_config = replace(
            base_config,
            initial_soc_mwh=soc,
            terminal_soc_mwh=terminal_soc,
        )

        remaining_discharge = None
        if annual_cycle_limit is not None:
            remaining_discharge = max(
                0.0,
                annual_cycle_limit * base_config.energy_mwh
                - discharged_mwh_ytd,
            )

        plan = optimize_battery_dispatch(
            forecast_row,
            horizon_config,
            max_discharge_mwh=remaining_discharge,
        )

        first = plan.iloc[0]
        actual_price = float(actual_prices.loc[target])
        charge_mw = float(first["charge_mw"])
        discharge_mw = float(first["discharge_mw"])
        gross_margin = actual_price * (discharge_mw - charge_mw)
        degradation_cost = (
            base_config.degradation_cost_per_mwh
            * (charge_mw + discharge_mw)
        )

        soc_start = soc
        soc = float(first["soc_end_mwh"])
        discharged_mwh_ytd += discharge_mw

        records.append({
            "forecast_origin_utc": origin,
            "target_timestamp_utc": target,
            "year_alberta": target_year,
            "actual_price": actual_price,
            "charge_mw": charge_mw,
            "discharge_mw": discharge_mw,
            "soc_start_mwh": soc_start,
            "soc_end_mwh": soc,
            "gross_revenue_cad": gross_margin,
            "degradation_cost_cad": degradation_cost,
            "net_profit_cad": gross_margin - degradation_cost,
        })

    return pd.DataFrame(records)


if RUN_WALK_FORWARD:
    actual_price_series = (
        master
        .set_index("timestamp_utc")["pool_price"]
        .sort_index()
    )

    linear_dispatch = run_receding_horizon(
        linear_predictions,
        actual_price_series,
        REFERENCE_BATTERY,
    )
    boosted_dispatch = run_receding_horizon(
        boosted_predictions,
        actual_price_series,
        REFERENCE_BATTERY,
    )

    display(boosted_dispatch.head())


,forecast_origin_utc,target_timestamp_utc,year_alberta,actual_price,charge_mw,discharge_mw,soc_start_mwh,soc_end_mwh,gross_revenue_cad,degradation_cost_cad,net_profit_cad
0,2020-01-01 07:00:00+00:00,2020-01-01 08:00:00+00:00,2020,29.43,0.0,0.0,100.0,100.0,0.0,0.0,0.0
1,2020-01-01 08:00:00+00:00,2020-01-01 09:00:00+00:00,2020,29.66,0.0,0.0,100.0,100.0,0.0,0.0,0.0
2,2020-01-01 09:00:00+00:00,2020-01-01 10:00:00+00:00,2020,28.19,0.0,0.0,100.0,100.0,0.0,0.0,0.0
3,2020-01-01 10:00:00+00:00,2020-01-01 11:00:00+00:00,2020,31.17,0.0,0.0,100.0,100.0,0.0,0.0,0.0
4,2020-01-01 11:00:00+00:00,2020-01-01 12:00:00+00:00,2020,31.80,0.0,0.0,100.0,100.0,0.0,0.0,0.0


## 8. Strategy comparison

Compare all strategies using identical timestamps and battery assumptions:

1. no dispatch;
2. trailing-threshold rule;
3. regularized linear forecast;
4. boosted forecast;
5. verified AESO forecast benchmark;
6. perfect-foresight upper bound.

The principal economic metric is:

$$
\text{Value Capture}
=
\frac{
\text{Realized Strategy Profit}
}{
\text{Perfect-Foresight Profit}
}
$$

Also report throughput, cycles, maximum drawdown, profit concentration, and annual stability.

In [9]:
# ---------------------------------------------------------------------
# Matched-period economic comparison
# ---------------------------------------------------------------------

def run_trailing_threshold_dispatch(source, config):
    """Causal rule using the prior price and trailing 30-day quartiles."""

    rule = source[
        ["timestamp_utc", "year_alberta", "pool_price"]
    ].copy()
    history = rule["pool_price"].shift(1)
    rule["signal_price"] = history
    rule["trailing_low"] = history.rolling(
        24 * 30,
        min_periods=24 * 7,
    ).quantile(0.25)
    rule["trailing_high"] = history.rolling(
        24 * 30,
        min_periods=24 * 7,
    ).quantile(0.75)

    records = []
    for _, year_frame in rule.groupby("year_alberta", sort=True):
        year_frame = year_frame.reset_index(drop=True)
        soc = config.resolved_initial_soc
        terminal_soc = config.resolved_terminal_soc

        for position, row in year_frame.iterrows():
            desired_soc = soc
            if pd.notna(row["trailing_low"]):
                if row["signal_price"] <= row["trailing_low"]:
                    desired_soc = min(
                        config.energy_mwh,
                        soc + config.charge_efficiency * config.power_mw,
                    )
                elif row["signal_price"] >= row["trailing_high"]:
                    desired_soc = max(
                        0.0,
                        soc - config.power_mw / config.discharge_efficiency,
                    )

            # Keep the terminal target physically reachable without using any
            # future price information. The final hour therefore returns SOC
            # exactly to the prespecified annual target.
            hours_after = len(year_frame) - position - 1
            reachable_low = max(
                0.0,
                terminal_soc
                - hours_after
                * config.charge_efficiency
                * config.power_mw,
            )
            reachable_high = min(
                config.energy_mwh,
                terminal_soc
                + hours_after
                * config.power_mw
                / config.discharge_efficiency,
            )
            next_soc = float(np.clip(
                desired_soc,
                reachable_low,
                reachable_high,
            ))

            soc_change = next_soc - soc
            charge_mw = (
                soc_change / config.charge_efficiency
                if soc_change > 0
                else 0.0
            )
            discharge_mw = (
                -soc_change * config.discharge_efficiency
                if soc_change < 0
                else 0.0
            )
            gross_margin = row["pool_price"] * (
                discharge_mw - charge_mw
            )
            degradation_cost = (
                config.degradation_cost_per_mwh
                * (charge_mw + discharge_mw)
            )

            records.append({
                "target_timestamp_utc": row["timestamp_utc"],
                "year_alberta": int(row["year_alberta"]),
                "actual_price": float(row["pool_price"]),
                "charge_mw": charge_mw,
                "discharge_mw": discharge_mw,
                "soc_start_mwh": soc,
                "soc_end_mwh": next_soc,
                "gross_revenue_cad": gross_margin,
                "degradation_cost_cad": degradation_cost,
                "net_profit_cad": gross_margin - degradation_cost,
            })
            soc = next_soc

    return pd.DataFrame(records)


def optimize_perfect_foresight_by_year(prices, config):
    frames = []
    local_year = prices.index.tz_convert("America/Edmonton").year
    for _, annual_prices in prices.groupby(local_year):
        annual = optimize_battery_dispatch(annual_prices, config)
        annual.index.name = "target_timestamp_utc"
        annual = annual.reset_index()
        frames.append(annual)
    return pd.concat(frames, ignore_index=True)


def summarize_realized_strategy(dispatch, config):
    if dispatch.empty:
        return pd.Series(dtype=float)

    cumulative_profit = dispatch["net_profit_cad"].cumsum()
    drawdown = cumulative_profit - cumulative_profit.cummax()
    positive_profit = dispatch["net_profit_cad"].clip(lower=0)
    positive_total = positive_profit.sum()
    top_count = max(1, int(np.ceil(0.01 * len(dispatch))))

    return pd.Series({
        "net_profit_cad": dispatch["net_profit_cad"].sum(),
        "charged_mwh": dispatch["charge_mw"].sum(),
        "discharged_mwh": dispatch["discharge_mw"].sum(),
        "equivalent_full_cycles": (
            dispatch["discharge_mw"].sum() / config.energy_mwh
        ),
        "maximum_drawdown_cad": -drawdown.min(),
        "top_1_pct_positive_profit_share": (
            positive_profit.nlargest(top_count).sum() / positive_total
            if positive_total > 0
            else np.nan
        ),
    })


def restrict_to_timestamps(dispatch, timestamps):
    return (
        dispatch.loc[
            dispatch["target_timestamp_utc"].isin(timestamps)
        ]
        .sort_values("target_timestamp_utc")
        .reset_index(drop=True)
    )


if RUN_WALK_FORWARD:
    threshold_dispatch = run_trailing_threshold_dispatch(
        master,
        REFERENCE_BATTERY,
    )

    matched_timestamps = set(
        linear_dispatch["target_timestamp_utc"]
    )
    matched_timestamps &= set(
        boosted_dispatch["target_timestamp_utc"]
    )
    matched_timestamps &= set(
        threshold_dispatch["target_timestamp_utc"]
    )
    matched_timestamps = pd.DatetimeIndex(
        sorted(matched_timestamps)
    )

    matched_prices = actual_price_series.loc[matched_timestamps]
    perfect_dispatch = optimize_perfect_foresight_by_year(
        matched_prices,
        REFERENCE_BATTERY,
    )

    strategy_dispatches = {
        "No dispatch": pd.DataFrame({
            "target_timestamp_utc": matched_timestamps,
            "charge_mw": 0.0,
            "discharge_mw": 0.0,
            "net_profit_cad": 0.0,
        }),
        "Trailing-threshold rule": restrict_to_timestamps(
            threshold_dispatch,
            matched_timestamps,
        ),
        "Linear forecast": restrict_to_timestamps(
            linear_dispatch,
            matched_timestamps,
        ),
        "Boosted forecast": restrict_to_timestamps(
            boosted_dispatch,
            matched_timestamps,
        ),
        "Perfect foresight": restrict_to_timestamps(
            perfect_dispatch,
            matched_timestamps,
        ),
    }

    comparison = pd.DataFrame({
        name: summarize_realized_strategy(dispatch, REFERENCE_BATTERY)
        for name, dispatch in strategy_dispatches.items()
    }).T

    perfect_profit = comparison.loc[
        "Perfect foresight",
        "net_profit_cad",
    ]
    comparison["value_capture_pct"] = (
        100 * comparison["net_profit_cad"] / perfect_profit
    )

    display(comparison.round(2))

    aeso_benchmark_status = pd.Series({
        "available_in_master": master[
            "aeso_forecast_pool_price_cad_mwh"
        ].notna().any(),
        "publication_timing_verified": False,
        "included_in_dispatch_comparison": False,
    })
    display(aeso_benchmark_status.to_frame("status"))


,net_profit_cad,charged_mwh,discharged_mwh,equivalent_full_cycles,maximum_drawdown_cad,top_1_pct_positive_profit_share,value_capture_pct
No dispatch,0.000000e+00,0.00,0.00,0.00,-0.00,NaN,0.00
Trailing-threshold rule,2.212572e+07,204300.41,179778.56,898.89,60712.32,0.67,22.03
Linear forecast,4.541282e+07,480278.59,422738.97,2113.69,117037.89,0.41,45.22
Boosted forecast,3.980647e+07,369309.85,324986.86,1624.93,157175.00,0.48,39.64
Perfect foresight,1.004165e+08,764092.20,672401.14,3362.01,170677.00,0.27,100.00


,status
available_in_master,True
publication_timing_verified,False
included_in_dispatch_comparison,False


## 9. Robustness and failure cases

The robustness stage separates **dispatch sensitivities**, which can reuse a
fixed set of price forecasts, from **information-set sensitivities**, which
require the forecasting models to be refit.

The configured tests cover:

- 12-, 24-, and 48-hour dispatch horizons;
- current-SOC versus half-full terminal targets;
- expanding versus three-year rolling training windows;
- 1-, 2-, and 4-hour batteries;
- 80%, 88%, and 92% round-trip efficiency;
- $0, $2, and $5/MWh degradation charges;
- annual cycle budgets;
- one- and six-hour publication delays;
- performance outside the 2021–2023 high-price regime; and
- the largest missed opportunities and loss-making dispatches.

These tests are deliberately gated by `RUN_ROBUSTNESS` because the hourly
receding-horizon simulations are computationally expensive.


In [10]:
# ---------------------------------------------------------------------
# Robustness grid and failure-case inspection
# ---------------------------------------------------------------------

robustness_register = pd.DataFrame([
    {"test": "Dispatch horizon", "values": "12, 24, 48 hours"},
    {"test": "Terminal SOC", "values": "hold current, half full"},
    {"test": "Training window", "values": "expanding, prior 3 years"},
    {"test": "Battery duration", "values": "1, 2, 4 hours"},
    {"test": "Round-trip efficiency", "values": "80%, 88%, 92%"},
    {"test": "Degradation cost", "values": "$0, $2, $5/MWh throughput"},
    {"test": "Annual cycle limit", "values": "250, 500, unlimited"},
    {"test": "Publication delay", "values": "0, 1, 6 hours"},
    {"test": "Market regime", "values": "all years, exclude 2021–2023"},
])
display(robustness_register)


def dispatch_sensitivity_row(
    label,
    value,
    predictions,
    actual_prices,
    config,
    **dispatch_kwargs,
):
    dispatch = run_receding_horizon(
        predictions,
        actual_prices,
        config,
        **dispatch_kwargs,
    )
    summary = summarize_realized_strategy(dispatch, config)
    return dispatch, {
        "test": label,
        "value": value,
        **summary.to_dict(),
    }


if RUN_ROBUSTNESS:
    actual_price_series = (
        master
        .set_index("timestamp_utc")["pool_price"]
        .sort_index()
    )

    # Fit 48 leads once; shorter dispatch horizons reuse the relevant leads.
    robust_predictions, _ = run_direct_forecast_evaluation(
        master,
        make_boosted_model,
        model_name="Boosted",
        max_lead_hours=48,
    )

    robustness_rows = []
    robustness_dispatches = {}

    for horizon in [12, 24, 48]:
        dispatch, row = dispatch_sensitivity_row(
            "Dispatch horizon",
            f"{horizon} hours",
            robust_predictions,
            actual_price_series,
            REFERENCE_BATTERY,
            horizon_hours=horizon,
        )
        robustness_rows.append(row)
        robustness_dispatches[("Dispatch horizon", horizon)] = dispatch

    for policy in ["hold_current", "half_full"]:
        _, row = dispatch_sensitivity_row(
            "Terminal SOC",
            policy,
            robust_predictions,
            actual_price_series,
            REFERENCE_BATTERY,
            horizon_hours=24,
            terminal_soc_policy=policy,
        )
        robustness_rows.append(row)

    one_at_a_time_configs = []
    for duration in [1.0, 2.0, 4.0]:
        one_at_a_time_configs.append((
            "Battery duration",
            f"{duration:g} hours",
            replace(
                REFERENCE_BATTERY,
                energy_mwh=REFERENCE_BATTERY.power_mw * duration,
                initial_soc_mwh=0.5 * REFERENCE_BATTERY.power_mw * duration,
                terminal_soc_mwh=0.5 * REFERENCE_BATTERY.power_mw * duration,
            ),
        ))
    for efficiency in [0.80, 0.88, 0.92]:
        one_at_a_time_configs.append((
            "Round-trip efficiency",
            f"{efficiency:.0%}",
            replace(
                REFERENCE_BATTERY,
                round_trip_efficiency=efficiency,
            ),
        ))
    for cost in [0.0, 2.0, 5.0]:
        one_at_a_time_configs.append((
            "Degradation cost",
            f"${cost:g}/MWh",
            replace(
                REFERENCE_BATTERY,
                degradation_cost_per_mwh=cost,
            ),
        ))

    for label, value, config in one_at_a_time_configs:
        _, row = dispatch_sensitivity_row(
            label,
            value,
            robust_predictions,
            actual_price_series,
            config,
            horizon_hours=24,
        )
        robustness_rows.append(row)

    for cycle_limit in [250, 500, None]:
        _, row = dispatch_sensitivity_row(
            "Annual cycle limit",
            "Unlimited" if cycle_limit is None else str(cycle_limit),
            robust_predictions,
            actual_price_series,
            REFERENCE_BATTERY,
            horizon_hours=24,
            annual_cycle_limit=cycle_limit,
        )
        robustness_rows.append(row)

    # Information-set tests refit the forecasts rather than reusing predictions.
    for window in [None, 3]:
        window_predictions, _ = run_direct_forecast_evaluation(
            master,
            make_boosted_model,
            model_name="Boosted",
            training_window_years=window,
        )
        _, row = dispatch_sensitivity_row(
            "Training window",
            "Expanding" if window is None else "Prior 3 years",
            window_predictions,
            actual_price_series,
            REFERENCE_BATTERY,
        )
        robustness_rows.append(row)

    for delay in [0, 1, 6]:
        delayed_predictions, _ = run_direct_forecast_evaluation(
            master,
            make_boosted_model,
            model_name="Boosted",
            publication_delay_hours=delay,
        )
        _, row = dispatch_sensitivity_row(
            "Publication delay",
            f"{delay} hours",
            delayed_predictions,
            actual_price_series,
            REFERENCE_BATTERY,
        )
        robustness_rows.append(row)

    robustness_results = pd.DataFrame(robustness_rows)
    display(robustness_results.round(2))

    base_dispatch = robustness_dispatches[("Dispatch horizon", 24)]
    outside_regime = base_dispatch.loc[
        ~base_dispatch["year_alberta"].between(2021, 2023)
    ]
    regime_stability = pd.DataFrame({
        "All evaluation years": summarize_realized_strategy(
            base_dispatch,
            REFERENCE_BATTERY,
        ),
        "Excluding 2021–2023": summarize_realized_strategy(
            outside_regime,
            REFERENCE_BATTERY,
        ),
    }).T
    display(regime_stability.round(2))

    matched_prices = actual_price_series.loc[
        base_dispatch["target_timestamp_utc"]
    ]
    perfect_for_failures = optimize_perfect_foresight_by_year(
        matched_prices,
        REFERENCE_BATTERY,
    )
    failure_cases = (
        base_dispatch[
            ["target_timestamp_utc", "actual_price", "net_profit_cad"]
        ]
        .rename(columns={"net_profit_cad": "strategy_profit_cad"})
        .merge(
            perfect_for_failures[
                ["target_timestamp_utc", "net_profit_cad"]
            ].rename(columns={"net_profit_cad": "perfect_profit_cad"}),
            on="target_timestamp_utc",
            how="inner",
            validate="one_to_one",
        )
    )
    failure_cases["opportunity_cost_cad"] = (
        failure_cases["perfect_profit_cad"]
        - failure_cases["strategy_profit_cad"]
    )

    display(
        failure_cases
        .nlargest(20, "opportunity_cost_cad")
        .round(2)
    )
    display(
        failure_cases
        .nsmallest(20, "strategy_profit_cad")
        .round(2)
    )
else:
    print(
        "Robustness tests are configured but not run. "
        "Set RUN_ROBUSTNESS = True after the core walk-forward run is stable."
    )


,test,values
0,Dispatch horizon,"12, 24, 48 hours"
1,Terminal SOC,"hold current, half full"
2,Training window,"expanding, prior 3 years"
3,Battery duration,"1, 2, 4 hours"
4,Round-trip efficiency,"80%, 88%, 92%"
5,Degradation cost,"$0, $2, $5/MWh throughput"
6,Annual cycle limit,"250, 500, unlimited"
7,Publication delay,"0, 1, 6 hours"
8,Market regime,"all years, exclude 2021–2023"


Robustness tests are configured but not run. Set RUN_ROBUSTNESS = True after the core walk-forward run is stable.


## 10. Final synthesis

### Main result

The **linear price forecast produced the highest realized battery value** among the causal strategies tested. Over the common 2020–2025 evaluation sample, it earned **$45.41 million** in degradation-adjusted net arbitrage profit and captured **45.22%** of the matched perfect-foresight value. The boosted forecast earned **$39.81 million** and captured **39.64%**, while the trailing-threshold rule earned **$22.13 million** and captured **22.03%**. Perfect foresight earned **$100.42 million** on the same timestamps, so a large operational information gap remained even for the strongest forecast-driven strategy.

### Forecast accuracy was not the same as economic value

Lower average price error did **not** translate automatically into higher arbitrage profit. The boosted model had lower MAE than the linear model in every evaluation year, but produced about **$5.61 million less** battery profit. Battery value depended more on correctly ranking and timing economically important spreads than on minimizing average hourly error. This distinction was most visible in the price tail: at a 24-hour lead, mean absolute error for $500–$900/MWh hours was approximately **$467/MWh** for the linear model and **$576/MWh** for the boosted model; for $900+/MWh hours it was approximately **$667/MWh** and **$804/MWh**, respectively. The linear model's better tail performance helps explain why it dispatched the battery more profitably despite its worse overall MAE.

### Stability and failure cases

Forecast accuracy was materially worse during the volatile 2021–2023 market regime and improved again in 2024–2025. Boosted-model MAE rose from **$17.38/MWh in 2020** to **$52.68–$88.31/MWh in 2021–2023**, before falling to **$40.76/MWh in 2024** and **$25.82/MWh in 2025**. The linear model followed the same broad pattern. This supports the conclusion that the forecasting problem became harder during the high-price regime, but the notebook does **not** yet report annual strategy profit or annual value capture. Economic stability outside 2021–2023 therefore remains unproven.

Opportunity cost was concentrated in the extreme-price tail, where both models substantially underpredicted scarcity as lead time increased. However, the current output does not rank individual scarcity episodes by forgone battery profit, so no specific event should be described as the dominant missed opportunity. That requires an event-level counterfactual attribution comparing forecast-driven and perfect-foresight dispatch.

### Robustness still required

The notebook configures tests for 12-, 24-, and 48-hour dispatch horizons and for hold-current versus half-full terminal-SOC policies, but those tests were not executed in the reported run. The sensitivity of value to forecast horizon and terminal SOC is therefore **not yet established**. The same qualification applies to rolling training windows, battery duration, efficiency, degradation cost, cycle limits, and publication delays.

### Information available at the forecast origin

The core models use target-hour calendar fields together with information shifted back to the forecast origin: prior pool prices, prior price statistics, and lagged net load, outages, imports, and gas price. Calendar variables are known in advance. The lagged market and system variables are only operationally valid to the extent that their publication delays are no longer than the modeled delay; this notebook applies an explicit delay convention but does not independently verify every source's historical publication timestamp. Same-hour realized system state and ERA5 reanalysis are excluded from the predictor set. The archived AESO pool-price forecast is retained only as an external benchmark because its issue and revision timing has not been verified, and it is not included in the dispatch comparison.

### Bottom line

The results are a credible historical proof of concept: a simple linear forecast converted prior-only market information into materially more arbitrage value than a causal threshold rule, while capturing almost half of the perfect-foresight upper bound. They also show why electricity-price forecasts should be evaluated economically rather than by average error alone. The remaining gap to perfect foresight, large tail errors, and incomplete timing and robustness audits mean the results are not yet evidence of live trading performance or investment-grade battery value.

This notebook is a historical walk-forward simulation under an explicit information set. It is not live trading performance, and archived forecasts should not be described as operationally available until their publication timing has been verified.
